# Mini Project 1: Cobblestone Gifts Cleaning Project


In [6]:
import kagglehub
import pandas as pd
import numpy as np
import re

ModuleNotFoundError: No module named 'kagglesdk.competitions.legacy'

## PHASE A · Load & profile
This section will load the raw dataset, inspect its shape and structure, and record the first data-quality issues that need attention.

In [ ]:
class DataLoader:
    def __init__(self, data_source):
        self.data_source = data_source
        self.path = None        
    
    def download_kaggle(self):
        """
        Downloads a dataset from Kaggle.

        :param dataset_name: The name of the dataset to download.
        """
        print(f"Downloading {self.data_source} from Kaggle...")
        self.path = kagglehub.dataset_download(self.data_source)
        return self.path

    def load_data(self):
        """
        Loads data from the specified data source.

        :return: Loaded data.
        """
        print(f"Loading data from {self.path}")
        self.df = pd.read_csv(f"{self.path}/data.csv", encoding='ISO-8859-1')
        # This section should report the dataset shape, inspect the first rows, and summarize the column types, missing values, and overall data quality using `head()`, `shape`, `info()`, and `describe()`.
        print(f"Dataset shape: {self.df.shape}")
        print("First 4 rows:")
        print(self.df.head(4))
        print(f"Data types and missing values:\n{self.df.info()}")
        print(f"Data quality summary:\n{self.df.describe()}")
        return self.df  
    
    def unique_count(self, columns):
        """
        Counts unique values in specified columns.
        
        :param columns: List of column names to analyze.
        """
        for column in columns:
            print(f"Total number of uique values in {column}: {self.df[column].nunique()}")
            print(f"Unique values and counts in {column}:\n{self.df[column].value_counts(dropna=False)}")
            
    def summary_statistics(self, columns):
        """
        Provides summary statistics for specified columns.
        
        :param columns: List of column names to analyze.
        """
        for column in columns:
            print(f"Summary statistics for {column}")
            print(f"Min: {self.df[column].min()}, Max: {self.df[column].max()}, Sum: {self.df[column].sum()}, Mean: {self.df[column].mean()}, Count: {self.df[column].count()}")
            
    
    def identify_negative_zero(self, columns):
        """
        Identifies negative and zero values in specified numeric columns.
        
        :param columns: List of numeric column names to analyze.
        """
        for column in columns:
            if pd.api.types.is_numeric_dtype(self.df[column]):
                negative_count = (self.df[column] < 0).sum()
                zero_count = (self.df[column] == 0).sum()
                print(f"Column {column}: Negative values count = {negative_count}, Zero values count = {zero_count}")
            else:
                print(f"Column {column} is not numeric and will be skipped.")

### 3.2 Getting information
Load the raw CSV with the correct encoding, then record shape, preview rows, data types, summary statistics, and missing-value counts for every column.

In [ ]:
# Loading carrie1/ecommerce-data
data_loader = DataLoader("carrie1/ecommerce-data")
data_loader.download_kaggle()
df = data_loader.load_data()
print()

Loading data from C:\Users\omarg\.cache\kagglehub\datasets\carrie1\ecommerce-data\versions\1
Dataset shape: (541909, 8)
First 4 rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 

### 3.9 Unique values
Use `value_counts()` and `nunique()` on Country, StockCode, and Description to identify non-product codes, missing labels, and unusual category patterns.

In [ ]:
data_loader.unique_count(['Country', 'StockCode', 'Description'])

Total number of uique values in Country: 38
Unique values and counts in Country:
Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab 

### 3.8 Min/max/sum/mean/count
Summarize Quantity and UnitPrice to identify negative quantities, zero prices, and other impossible values that cannot represent valid sales.

In [ ]:
data_loader.summary_statistics(['Quantity', 'UnitPrice'])

Summary statistics for Quantity
Min: -80995, Max: 80995, Sum: 5176450, Mean: 9.55224954743324, Count: 541909
Summary statistics for UnitPrice
Min: -11062.06, Max: 38970.0, Sum: 2498803.9740000004, Mean: 4.611113626088514, Count: 541909


### 3.1 Creating a DataFrame
Build a small country → region lookup table from a Python dictionary (e.g. United
Kingdom → UK\&IE; France, Germany → Western Europe). 

In [ ]:
region_map = {
    'UK&IE': ['United Kingdom', 'Ireland'],
    'Western Europe': ['France', 'Germany', 'Belgium'],
    'Nordics': ['Sweden', 'Norway', 'Denmark']
}

lookup_table = pd.DataFrame([
    {'Country': country, 'Region': region}
    for region, countries in region_map.items()
    for country in countries
])

## PHASE B · Slice, filter & sort
This section explores the raw DataFrame using positional and label-based indexing, conditional filters, and sorting.

In [ ]:
class DataExplorer:
    """Phase B — slice, filter, and sort the raw DataFrame."""

    def add_line_item_id(self, df):
        """3.3 Add a LineItemID index (L0, L1, ...) for label-based access."""
        df = df.copy()
        df["LineItemID"] = ["L" + str(i) for i in range(len(df))]
        return df.set_index("LineItemID")

    def slice_iloc(self, df):
        """3.3 Positional slices: first 5 rows, rows 100-110, last 5, and a 5x4 block."""
        print("First 5 rows:"); print(df.iloc[0:5])
        print("Rows 100-110:"); print(df.iloc[100:110])
        print("Last 5 rows:"); print(df.iloc[-5:])
        print("First 5 rows, first 4 columns:"); print(df.iloc[0:5, 0:4])

    def slice_loc(self, df):
        """3.3 Label-based access: single key, range, list, and boolean mask."""
        print(df.loc["L100"])
        print(df.loc["L100":"L105"])
        print(df.loc[["L0", "L50", "L999"], ["StockCode", "Description", "Quantity"]])
        print(df.loc[df["Quantity"] < 0, ["InvoiceNo", "StockCode", "Quantity"]])

    def filter(self, df):
        """3.4 Conditional selection: cancellations, non-positive quantity, non-positive price."""
        cancellations = df[df["InvoiceNo"].astype(str).str.startswith("C")]
        print(f"Cancellations (InvoiceNo starts with C): {len(cancellations)}")
        qty_non_positive = df[df["Quantity"] <= 0]
        print(f"Quantity <= 0: {len(qty_non_positive)}")
        price_non_positive = df[df["UnitPrice"] <= 0]
        print(f"UnitPrice <= 0: {len(price_non_positive)}")

    def sort(self, df):
        """3.5 Sort by Quantity and LineRevenue to surface bulk orders and large returns."""
        df = df.copy()
        df["LineRevenue"] = df["Quantity"] * df["UnitPrice"]
        cols = ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "LineRevenue"]
        print("Largest bulk orders (by Quantity):")
        print(df.sort_values("Quantity", ascending=False)[cols].head(10))
        print("\nLargest returns (by Quantity):")
        print(df.sort_values("Quantity", ascending=True)[cols].head(10))
        print("\nLargest line revenue:")
        print(df.sort_values("LineRevenue", ascending=False)[cols].head(10))
        print("\nLargest revenue losses:")
        print(df.sort_values("LineRevenue", ascending=True)[cols].head(10))
        return df


### 3.3 Slicing

In [ ]:
explorer = DataExplorer()
indexed_df = explorer.add_line_item_id(df)
explorer.slice_iloc(indexed_df)


First 5 rows:
           InvoiceNo StockCode                          Description  Quantity  \
LineItemID                                                                      
L0            536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
L1            536365     71053                  WHITE METAL LANTERN         6   
L2            536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
L3            536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
L4            536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

               InvoiceDate  UnitPrice  CustomerID         Country  
LineItemID                                                         
L0          12/1/2010 8:26       2.55     17850.0  United Kingdom  
L1          12/1/2010 8:26       3.39     17850.0  United Kingdom  
L2          12/1/2010 8:26       2.75     17850.0  United Kingdom  
L3          12/1/2010 8:26       3.39     17850.0  United Kingdom  
L4        

In [ ]:
explorer.slice_loc(indexed_df)


InvoiceNo                               536378
StockCode                               84519A
Description    TOMATO CHARLIE+LOLA COASTER SET
Quantity                                     6
InvoiceDate                     12/1/2010 9:37
UnitPrice                                 2.95
CustomerID                             14688.0
Country                         United Kingdom
Name: L100, dtype: object
           InvoiceNo StockCode                          Description  Quantity  \
LineItemID                                                                      
L100          536378    84519A      TOMATO CHARLIE+LOLA COASTER SET         6   
L101          536378    85183B  CHARLIE & LOLA WASTEPAPER BIN FLORA        48   
L102          536378    85071B   RED CHARLIE+LOLA PERSONAL DOORSIGN        96   
L103          536378     21931               JUMBO STORAGE BAG SUKI        10   
L104          536378     21929       JUMBO BAG PINK VINTAGE PAISLEY        10   
L105          536380     22961 

### 3.4 Conditional Selection

In [ ]:
explorer.filter(indexed_df)


Cancellations (InvoiceNo starts with C): 9288
Quantity <= 0: 10624
UnitPrice <= 0: 2517


### 3.5 Sorting Values

In [ ]:
indexed_df = explorer.sort(indexed_df)


Largest bulk orders (by Quantity):
           InvoiceNo StockCode                          Description  Quantity  \
LineItemID                                                                      
L540421       581483     23843          PAPER CRAFT , LITTLE BIRDIE     80995   
L61619        541431     23166       MEDIUM CERAMIC TOP STORAGE JAR     74215   
L502122       578841     84826       ASSTD DESIGN 3D PAPER STICKERS     12540   
L74614        542504     37413                                  NaN      5568   
L421632       573008     84077    WORLD WAR 2 GLIDERS ASSTD DESIGNS      4800   
L206121       554868     22197                 SMALL POPCORN HOLDER      4300   
L220843       556231    85123A                                    ?      4000   
L97432        544612     22053                EMPIRE DESIGN ROSETTE      3906   
L270885       560599     18007  ESSENTIAL BALM 3.5g TIN IN ENVELOPE      3186   
L160546       550461     21108   FAIRY CAKE FLANNEL ASSORTED COLOUR      3

## Phase C Clean and Fix


In [ ]:
class DataCleaner:
    """Phase C — one method per cleaning step, each callable independently."""

    def replace_values(self, df, column, replacements):
        """3.6 Replace specific values in a column."""
        df = df.copy()
        df[column] = df[column].replace(replacements)
        return df

    def rename_columns(self, df, overrides=None):
        """3.7 Rename all columns to snake_case, with optional explicit overrides."""
        df = df.copy()
        def to_snake(name):
            name = re.sub(r'(?<!^)(?=[A-Z])', '_', name)
            return name.lower()
        df = df.rename(columns={col: to_snake(col) for col in df.columns})
        if overrides:
            df = df.rename(columns=overrides)
        return df

    def handle_missing(self, df, drop_cols=None, fill_map=None):
        """3.10 Drop rows missing required columns; fill optional columns."""
        df = df.copy()
        if drop_cols:
            for col in drop_cols:
                if col in df.columns and (pd.api.types.is_string_dtype(df[col])
                                          or df[col].dtype == object):
                    df[col] = df[col].str.strip().replace('', np.nan)
            df = df.dropna(subset=drop_cols)
        for col, val in (fill_map or {}).items():
            if col in df.columns:
                df[col] = df[col].fillna(val)
        return df

    def drop_columns(self, df, columns):
        """3.11 Delete one or more columns."""
        df = df.copy()
        return df.drop(columns=[c for c in columns if c in df.columns])

    def delete_rows(self, df, invoice_col='invoice_no', stock_col='stock_code',
                    qty_col='quantity', price_col='unit_price',
                    non_product_codes=('POST', 'DOT', 'M', 'BANK CHARGES', 'AMAZONFEE')):
        """3.12 Drop rows to build a clean sales DataFrame"""
        df = df.copy()
        # cancellations
        df = df[~df[invoice_col].astype(str).str.startswith('C')]
        # non-product lines: drop the named admin/non-product stock codes
        blacklist = {c.strip().upper() for c in non_product_codes}
        stock_norm = df[stock_col].astype(str).str.strip().str.upper()
        df = df[~stock_norm.isin(blacklist)]
        # impossible quantities / prices
        df = df[(df[qty_col] > 0) & (df[price_col] > 0)]
        return df

    def drop_duplicates(self, df):
        """3.13 Detect and remove exact duplicate rows; report how many were dropped."""
        df = df.copy()
        n_dupes = df.duplicated().sum()
        out = df.drop_duplicates()
        print(f"Exact duplicate rows: {n_dupes:,}")
        print(f"Rows before: {len(df):,}, after: {len(out):,} (dropped {len(df) - len(out):,})")
        return out


cleaner = DataCleaner()


### 3.6 Replacing Values

In [ ]:
# Keep the raw DataFrame untouched; build the clean version as a new object.
raw_df = df.copy()     # immutable snapshot of the raw export
work_df = df.copy()    # working copy we clean step by step

# Derived helper used during cleaning/exploration; dropped from the clean table in 3.11.
work_df["LineRevenue"] = work_df["Quantity"] * work_df["UnitPrice"]

# 3.6 Standardise country labels
work_df = cleaner.replace_values(work_df, column='Country', replacements={
    'EIRE': 'Ireland',
    'RSA': 'South Africa',
    'Unspecified': np.nan,
})

print(work_df['Country'].value_counts(dropna=False))


Country
United Kingdom          495478
Germany                   9495
France                    8557
Ireland                   8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
NaN                        446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
South Africa                58


### 3.7 Renaming Columns

In [ ]:
work_df = cleaner.rename_columns(work_df, overrides={'customer_i_d': 'customer_id'})

print(work_df.columns.tolist())


['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'line_revenue']


### 3.10 Handling Missing Values

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': work_df.isnull().sum(),
    'missing_pct': (work_df.isnull().sum() / len(work_df) * 100).round(2)
})
print(missing_summary[missing_summary['missing_count'] > 0])

work_df = cleaner.handle_missing(work_df,
    drop_cols=['description'],
    fill_map={'customer_id': 'Unknown'},
)

print("\nAfter handling:")
print(work_df.isnull().sum()[work_df.isnull().sum() > 0])
print("\nFor customer-level analysis later, exclude unknown buyers, e.g.:")
print("    customer_df = clean_sales_df[clean_sales_df['customer_id'] != 'Unknown'].copy()")


             missing_count  missing_pct
description           1454         0.27
customer_id         135080        24.93
country                446         0.08

After handling:
country    446
dtype: int64

For customer-level analysis later, exclude unknown buyers, e.g.:
    customer_df = clean_sales_df[clean_sales_df['customer_id'] != 'Unknown'].copy()


### 3.11 Deleting a Column

In [ ]:
work_df = cleaner.drop_columns(work_df, columns=['line_revenue'])

print(work_df.columns.tolist())


['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country']


Justification:

line_revenue was removed as it is a derived column from [quantity * unit_price] it is not an original data soruce, and the information is recoverable from those 2 columns 

### 3.12 Deleting a row
Remove cancellations, non-product lines, and impossible prices/quantities to build a clean sales DataFrame.

In [ ]:
before = len(work_df)
clean_sales_df = cleaner.delete_rows(work_df)
after = len(clean_sales_df)

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Removed:     {before - after:,}")

print("\nClean sales summary:")
print(clean_sales_df[['quantity', 'unit_price']].describe())


Rows before: 540,455
Rows after:  527,936
Removed:     12,519

Clean sales summary:
            quantity     unit_price
count  527936.000000  527936.000000
mean       10.564313       3.299968
std       155.808242      15.857034
min         1.000000       0.001000
25%         1.000000       1.250000
50%         3.000000       2.080000
75%        11.000000       4.130000
max     80995.000000   11062.060000


In [ ]:
print("\nCancellations remaining:", clean_sales_df['invoice_no'].astype(str).str.startswith('C').sum())
print("Non-positive quantity:  ", (clean_sales_df['quantity'] <= 0).sum())
print("Non-positive unit_price:", (clean_sales_df['unit_price'] <= 0).sum())


Cancellations remaining: 0
Non-positive quantity:   0
Non-positive unit_price: 0


### 3.13 Dropping duplicates
Detect exact duplicate rows with `duplicated().sum()` and remove them with `drop_duplicates()`, reporting how many were dropped.

In [ ]:
clean_sales_df = cleaner.drop_duplicates(clean_sales_df)


Exact duplicate rows: 5,221
Rows before: 527,936, after: 522,715 (dropped 5,221)


## PHASE D · Engineer & summarize
This section creates analysis features, groups the cleaned data, and produces the summaries needed for charts and business questions.

In [ ]:
class DataSummarizer:
    """Phase D — engineer analysis features and summarize the cleaned sales data."""

    def apply_functions(self, df):
        """3.18 Add revenue, clean descriptions with apply, and flag cancelled invoices."""
        out = df.copy()

        # Required engineered feature: revenue = quantity * unit_price
        out["revenue"] = out["quantity"] * out["unit_price"]

        def clean_description(desc):
            """Strip extra spaces and convert product descriptions to title case."""
            if pd.isna(desc) or str(desc).strip() == "":
                return "Unknown Product"
            return str(desc).strip().title()

        # Required use of apply on Description
        out["description_clean"] = out["description"].apply(clean_description)

        # Required use of apply to flag cancelled invoices
        out["is_cancelled_invoice"] = out["invoice_no"].apply(
            lambda invoice: str(invoice).startswith("C")
        )

        return out

    def loop_over_column(self, df, n=10):
        """3.17 Demonstrate a normal loop and a list comprehension over Description."""
        sample_descriptions = df["description"].head(n)

        # Standard Python for-loop over a column sample
        loop_cleaned = []
        for desc in sample_descriptions:
            if pd.isna(desc) or str(desc).strip() == "":
                loop_cleaned.append("Unknown Product")
            else:
                loop_cleaned.append(str(desc).strip().title())

        # Equivalent list comprehension over the same sample
        list_comp_cleaned = [
            "Unknown Product" if pd.isna(desc) or str(desc).strip() == ""
            else str(desc).strip().title()
            for desc in sample_descriptions
        ]

        return pd.DataFrame({
            "original_description": sample_descriptions.values,
            "for_loop_result": loop_cleaned,
            "list_comprehension_result": list_comp_cleaned,
        })

    def group_by_values(self, df):
        """3.14 Use groupby to compute revenue by country and orders per customer."""
        revenue_by_country = (
            df.groupby("country", dropna=False)["revenue"]
              .sum()
              .sort_values(ascending=False)
              .reset_index(name="total_revenue")
        )

        customer_rows = df[df["customer_id"] != "Unknown"].copy()
        orders_per_customer = (
            customer_rows.groupby("customer_id")["invoice_no"]
                         .nunique()
                         .sort_values(ascending=False)
                         .reset_index(name="order_count")
        )

        return revenue_by_country, orders_per_customer

    def aggregate_statistics(self, df):
        """3.16 Use agg to calculate multiple statistics per country and per customer."""
        # First summarize to one row per invoice so mean order value is based on invoices,
        # not individual line items.
        invoice_level = (
            df.groupby(["invoice_no", "country", "customer_id"], dropna=False)
              .agg(
                  order_revenue=("revenue", "sum"),
                  line_count=("stock_code", "count"),
                  first_invoice_date=("invoice_date", "min")
              )
              .reset_index()
        )

        country_agg = (
            invoice_level.groupby("country", dropna=False)
                         .agg(
                             total_revenue=("order_revenue", "sum"),
                             mean_order_value=("order_revenue", "mean"),
                             median_order_value=("order_revenue", "median"),
                             transaction_count=("invoice_no", "nunique"),
                             line_count=("line_count", "sum")
                         )
                         .sort_values("total_revenue", ascending=False)
                         .reset_index()
        )

        customer_invoice_level = invoice_level[invoice_level["customer_id"] != "Unknown"].copy()
        customer_agg = (
            customer_invoice_level.groupby("customer_id")
                                  .agg(
                                      total_revenue=("order_revenue", "sum"),
                                      mean_order_value=("order_revenue", "mean"),
                                      median_order_value=("order_revenue", "median"),
                                      transaction_count=("invoice_no", "nunique"),
                                      line_count=("line_count", "sum")
                                  )
                                  .sort_values("total_revenue", ascending=False)
                                  .reset_index()
        )

        return invoice_level, country_agg, customer_agg

    def apply_to_groups(self, df):
        """3.19 Use groupby(...).apply(...) for a custom per-customer summary."""
        customer_df = df[df["customer_id"] != "Unknown"].copy()
        customer_df["invoice_date"] = pd.to_datetime(customer_df["invoice_date"], errors="coerce")

        def custom_customer_summary(group):
            invoice_totals = group.groupby("invoice_no")["revenue"].sum()
            return pd.Series({
                "total_spend": group["revenue"].sum(),
                "order_count": group["invoice_no"].nunique(),
                "active_months": group["invoice_date"].dt.to_period("M").nunique(),
                "first_purchase": group["invoice_date"].min(),
                "last_purchase": group["invoice_date"].max(),
                "avg_order_value": invoice_totals.mean(),
                "largest_order_value": invoice_totals.max()
            })

        customer_summary = (
            customer_df.groupby("customer_id")[["invoice_no", "revenue", "invoice_date"]]
                       .apply(custom_customer_summary)
                       .sort_values("total_spend", ascending=False)
                       .reset_index()
        )

        return customer_summary

    def group_by_time(self, df):
        """3.15 Parse invoice_date, set it as index, and resample weekly/monthly revenue."""
        time_df = df.copy()
        time_df["invoice_date"] = pd.to_datetime(time_df["invoice_date"], errors="coerce")
        time_df = time_df.dropna(subset=["invoice_date"]).set_index("invoice_date").sort_index()

        monthly_revenue = (
            time_df["revenue"]
            .resample("ME")
            .sum()
            .reset_index(name="monthly_revenue")
        )

        weekly_revenue = (
            time_df["revenue"]
            .resample("W")
            .sum()
            .reset_index(name="weekly_revenue")
        )

        return time_df, monthly_revenue, weekly_revenue


summarizer = DataSummarizer()

### 3.18 Applying a function
Create a revenue column and use `apply` to clean Description values and flag cancelled invoices.

In [ ]:
# Use the cleaned completed-sales dataframe from Phase C.
# This keeps the raw dataframe untouched and creates a new Part D dataframe.
part_d_df = summarizer.apply_functions(clean_sales_df)

print("Part D dataframe shape:", part_d_df.shape)
print("Total completed-sales revenue: £{:,.2f}".format(part_d_df["revenue"].sum()))
print("Cancelled invoices still present:", part_d_df["is_cancelled_invoice"].sum())

part_d_df[[
    "invoice_no", "description", "description_clean",
    "quantity", "unit_price", "revenue", "is_cancelled_invoice"
]].head(10)


### 3.17 Looping over a column
Show a short loop or list comprehension over one column, then note why `apply` is usually the clearer and faster option.

In [ ]:
description_loop_demo = summarizer.loop_over_column(part_d_df, n=10)
description_loop_demo


### 3.14 Grouping by values
Use `groupby` to calculate revenue by country and order counts per customer.

In [ ]:
revenue_by_country, orders_per_customer = summarizer.group_by_values(part_d_df)

print("Revenue by country using groupby:")
display(revenue_by_country.head(10))

print("\nNumber of orders per known customer using groupby:")
display(orders_per_customer.head(10))


### 3.16 Aggregating
Use `agg` to calculate several statistics at once, such as total revenue, mean order value, and transaction counts by country and customer.

In [ ]:
invoice_level, country_agg, customer_agg = summarizer.aggregate_statistics(part_d_df)

print("Country-level aggregation using agg:")
display(country_agg.head(10))

print("\nCustomer-level aggregation using agg:")
display(customer_agg.head(10))


### 3.19 Applying to groups
Use `groupby(...).apply(...)` for a custom per-customer summary such as total spend and active months.

In [ ]:
customer_summary_apply = summarizer.apply_to_groups(part_d_df)

print("Custom per-customer summary using groupby(...).apply(...):")
customer_summary_apply.head(10)


### 3.15 Grouping by time
Parse InvoiceDate to datetime, set it as the index, and resample revenue weekly and monthly to show the trend over time.

In [ ]:
import matplotlib.pyplot as plt

time_indexed_sales, monthly_revenue, weekly_revenue = summarizer.group_by_time(part_d_df)

print("Monthly revenue trend:")
display(monthly_revenue)

print("\nWeekly revenue trend sample:")
display(weekly_revenue.head(10))

plt.figure(figsize=(10, 5))
plt.plot(monthly_revenue["invoice_date"], monthly_revenue["monthly_revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(weekly_revenue["invoice_date"], weekly_revenue["weekly_revenue"])
plt.title("Weekly Revenue Trend")
plt.xlabel("Week")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## PHASE E · Combine
This section recombines the cleaned data and enriches it with the region lookup table used later in the analysis.

### 3.20 Concatenating
Split the cleaned data into two periods, then concatenate them back together to simulate combining separate extracts.

### 3.21 Merging
Merge the region lookup table into the cleaned data, then compare inner and left joins to note which countries do not match.